In [3]:
import os
import joblib    
import numpy  as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer


In [4]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.2 MB/s eta 0:00:00:00:0100:01
^C
ERROR: Operation cancelled by user


In [5]:
# load data
print("Loading data …")
df = pd.read_csv("/kaggle/input/datasets/rubinanoor/data-a2/data-a2.csv", low_memory=False)
print(f"  Raw shape: {df.shape}")


Loading data …
  Raw shape: (62302, 14)


In [6]:

# handle missing values 
df.dropna(subset=["price"], inplace=True)

df.dropna(subset=["year"], inplace=True)

# fill engine with median
df["engine"] = pd.to_numeric(df["engine"], errors="coerce")
df["engine"].fillna(df["engine"].median())

# fill fuel and body with mode (most common value)
df["fuel"] = df["fuel"].fillna(df["fuel"].mode()[0])
df["body"] = df["body"].fillna(df["body"].mode()[0])

print(f"  After null handling: {df.shape}")

  After null handling: (58171, 14)


In [7]:
# feature engineering
CURRENT_YEAR = 2024
df["car_age"] = CURRENT_YEAR - df["year"].astype(int)

df["mileage_per_year"] = df["mileage"] / (df["car_age"] + 1)

# feature selection 
FEATURES = ["car_age", "engine", "mileage", "mileage_per_year",
            "transmission", "fuel", "body", "city"]

# encode categorical variables

label_encoders = {}
cat_cols = ["transmission", "fuel", "body", "city"]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna("Unknown") 
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# target Variable 
median_price = df["price"].median()
df["price_category"] = (df["price"] >= median_price).astype(int)
print(f"  Median price: {median_price:,.0f} PKR")
print(f"  Class distribution:\n{df['price_category'].value_counts()}")



  Median price: 2,700,000 PKR
  Class distribution:
price_category
1    29235
0    28936
Name: count, dtype: int64


In [8]:
# build X and y 
X = df[FEATURES].copy()
y = df["price_category"]

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
print(f"\n  Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")


  Train size: 46,536  |  Test size: 11,635


In [9]:
# build pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # safety net for any NaN
    ("scaler",  StandardScaler()),
    ("svm",     SVC(kernel="rbf",
                    C=1.0,
                    gamma="scale",           # 'scale' = 1/(n_features * X.var())
                    probability=True,        # needed for predict_proba()
                    random_state=42)),
])

print("\nTraining SVM ")
pipeline.fit(X_train, y_train)
print("Training complete.")


Training SVM 
Training complete.


In [11]:
# evaluate
y_pred = pipeline.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                             target_names=["Low Price", "High Price"]))


Test Accuracy: 0.9337 (93.37%)

Classification Report:
              precision    recall  f1-score   support

   Low Price       0.93      0.94      0.93      5788
  High Price       0.94      0.93      0.93      5847

    accuracy                           0.93     11635
   macro avg       0.93      0.93      0.93     11635
weighted avg       0.93      0.93      0.93     11635



In [12]:
#save artefact
model_artefact = {
    "pipeline"       : pipeline,
    "label_encoders" : label_encoders,
    "features"       : FEATURES,
    "median_price"   : median_price,
    "current_year"   : CURRENT_YEAR,
}

joblib.dump(model_artefact, "pakwheels_svm_model.pkl")
print("\n Model saved: pakwheels_svm_model.pkl")
print(f"   Artefact keys: {list(model_artefact.keys())}")


 Model saved: pakwheels_svm_model.pkl
   Artefact keys: ['pipeline', 'label_encoders', 'features', 'median_price', 'current_year']


## PART2 TASK2

In [1]:
from typing import Optional
from contextlib import asynccontextmanager
from fastapi             import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware  # allows Streamlit to call us
from pydantic            import BaseModel, Field, field_validator
import numpy as np
import pandas as pd


In [2]:
MODEL_PATH = "/kaggle/working/pakwheels_svm_model.pkl"
model_artefact = None  

# lifespan handler
@asynccontextmanager
async def lifespan(app: FastAPI):
    """Handles startup and shutdown logic in one place."""
    global model_artefact
    
    
    print("Checking for model file...")
    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: Model file not found at {MODEL_PATH}")
    else:
        try:
            model_artefact = joblib.load(MODEL_PATH)
            print(f"Successfully loaded model from {MODEL_PATH}")
            print(f"Features expected: {model_artefact.get('features', 'Unknown')}")
        except Exception as e:
            print(f"Failed to load model: {e}")

    yield  #  running 
    
    # shutdown
    print("Shutting down API: Cleaning up resources...")

# initialize FastAPI with the lifespan
app = FastAPI(
    title       = "PakWheels Price Category API",
    description = "Predicts whether a used car is HIGH or LOW price using a trained SVM.",
    version     = "1.0.0",
    lifespan    = lifespan
)

# 4. Middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  
    allow_headers=["*"],
)

In [3]:
class CarFeatures(BaseModel):

    year: int = Field(
        ...,
        ge=1980, le=2024,
        description="Year of manufacture (1980–2024)",
        examples=[2018], # V2 uses 'examples' list
    )
    engine: float = Field(
        ...,
        ge=600, le=8000,
        description="Engine capacity in cc (600–8000)",
        examples=[1300],
    )
    mileage: int = Field(
        ...,
        ge=0, le=500_000,
        description="Odometer reading in km (0–500,000)",
        examples=[45000],
    )
    transmission: str = Field(
        ...,
        description="Transmission type: 'Manual' or 'Automatic'",
        examples=["Manual"],
    )
    fuel: str = Field(
        ...,
        description="Fuel type: 'Petrol', 'Diesel', 'Hybrid', or 'CNG'",
        examples=["Petrol"],
    )
    body: Optional[str] = Field(
        default="Sedan",
        description="Body type (optional)",
        examples=["Sedan"],
    )
    city: Optional[str] = Field(
        default="Karachi",
        description="City where the car is listed (optional)",
        examples=["Karachi"],
    )

    # Modern V2 Validator
    @field_validator("transmission")
    @classmethod 
    def normalise_transmission(cls, v: str) -> str:
        v = v.strip().title()
        if v not in ("Manual", "Automatic"):
            raise ValueError("transmission must be 'Manual' or 'Automatic'")
        return v

    @field_validator("fuel")
    @classmethod
    def normalise_fuel(cls, v: str) -> str:
        v = v.strip().title()
       
        if v == "Cng": v = "CNG" 
        if v not in ("Petrol", "Diesel", "Hybrid", "CNG", "Electric"):
            raise ValueError("fuel must be one of: Petrol, Diesel, Hybrid, CNG, Electric")
        return v


class PredictionResponse(BaseModel):
    price_category  : str   = Field(..., description="'High Price' or 'Low Price'")
    probability_high: float = Field(..., description="Confidence score for High Price class")
    probability_low : float = Field(..., description="Confidence score for Low Price class")
    median_price_pkr: float = Field(..., description="Threshold price (PKR) used for categorisation")
    input_received  : dict  = Field(..., description="Echo of the normalised input features")

In [4]:
# 4. HELPER — Feature Engineering & Encoding

def prepare_features(car: CarFeatures) -> pd.DataFrame:
  
    artefact      = model_artefact
    label_encoders = artefact["label_encoders"]
    features       = artefact["features"]
    current_year   = artefact["current_year"]

    car_age          = current_year - car.year
    mileage_per_year = car.mileage / (car_age + 1)   # +1 avoids div-by-zero

   
    raw = {
        "car_age"         : car_age,
        "engine"          : car.engine,
        "mileage"         : car.mileage,
        "mileage_per_year": mileage_per_year,
        "transmission"    : car.transmission,
        "fuel"            : car.fuel,
        "body"            : car.body or "Sedan",
        "city"            : car.city or "Karachi",
    }

    df_row = pd.DataFrame([raw])

    for col in ["transmission", "fuel", "body", "city"]:
        le = label_encoders.get(col)
        if le is not None:
            val = df_row[col].iloc[0]
            if val in le.classes_:
                df_row[col] = le.transform([val])
            else:
                df_row[col] = 0

    return df_row[features]

In [5]:
#endpoints

@app.get("/", tags=["Root"])
def root():
    return {"message": "PakWheels Price Category API is running. Visit /docs for usage."}


@app.get("/health", tags=["Health"])
def health_check():
    model_loaded = model_artefact is not None
    return {
        "status"      : "healthy" if model_loaded else "unhealthy",
        "model_loaded": model_loaded,
    }


@app.post("/predict", response_model=PredictionResponse, tags=["Prediction"])
def predict(car: CarFeatures):
   
    if model_artefact is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet. Retry in a moment.")

    try:
        X = prepare_features(car)

        pipeline      = model_artefact["pipeline"]
        pred_class    = pipeline.predict(X)[0]         # 0=Low, 1=High
        pred_proba    = pipeline.predict_proba(X)[0]   # [P(Low), P(High)]
        median_price  = model_artefact["median_price"]

        label = "High Price" if pred_class == 1 else "Low Price"

        return PredictionResponse(
            price_category   = label,
            probability_high = round(float(pred_proba[1]), 4),
            probability_low  = round(float(pred_proba[0]), 4),
            median_price_pkr = float(median_price),
            input_received   = car.dict(),
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")


@app.get("/model-info", tags=["Metadata"])
def model_info():
    """
    Returns metadata about the currently loaded model.
    Useful for debugging and logging in production.
    """
    if model_artefact is None:
        raise HTTPException(status_code=503, detail="Model not loaded.")
    return {
        "features"        : model_artefact["features"],
        "median_price_pkr": model_artefact["median_price"],
        "current_year"    : model_artefact["current_year"],
    }



In [6]:
if __name__ == "__main__":
    import uvicorn
    import asyncio
    import nest_asyncio

    # Allow nested loops
    nest_asyncio.apply()

    # Define the config separately
    config = uvicorn.Config(
        app=app, 
        host="0.0.0.0", 
        port=8000, 
        loop="asyncio"
    )
    
    # Create the server instance
    server = uvicorn.Server(config)

    # Use the existing notebook loop to run the server
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())
    
    print("API is running on http://0.0.0.0:8000")

API is running on http://0.0.0.0:8000


INFO:     Started server process [537]
INFO:     Waiting for application startup.
ERROR:    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/starlette/routing.py", line 694, in lifespan
    async with self.lifespan_context(app) as maybe_state:
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/contextlib.py", line 210, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_537/1086250974.py", line 12, in lifespan
    if not os.path.exists(MODEL_PATH):
           ^^
NameError: name 'os' is not defined. Did you forget to import 'os'

ERROR:    Application startup failed. Exiting.


Checking for model file...


## PART 2 TASK 3


In [17]:
%%writefile app.py

import json
import streamlit as st    
import requests          

# page config
st.set_page_config(
    page_title = "PakWheels Price Predictor",
    page_icon  = "🚗",
    layout     = "centered",
)

BACKEND_URL = "http://localhost:8000"   

TRANSMISSIONS = ["Manual", "Automatic"]
FUELS         = ["Petrol", "Diesel", "Hybrid", "CNG"]
BODIES        = ["Sedan", "Hatchback", "SUV", "Crossover", "Compact SUV",
                 "Van", "MPV", "Pickup", "Coupe"]
CITIES        = ["Karachi", "Lahore", "Islamabad", "Rawalpindi", "Faisalabad",
                 "Multan", "Peshawar", "Quetta", "Sialkot", "Gujranwala",
                 "Other"]

def call_predict_api(payload: dict) -> dict:

    try:
        response = requests.post(
            url     = f"{BACKEND_URL}/predict",
            json    = payload,
            timeout = 10,  
        )
        response.raise_for_status()  
        return response.json()

    except requests.exceptions.ConnectionError:
        st.error(
            "Cannot connect to the backend server.\n\n"
            f"Make sure FastAPI is running at `{BACKEND_URL}`.\n"
            "Command: `uvicorn Part2_Task2_backend:app --port 8000 --reload`"
        )
        return None

    except requests.exceptions.HTTPError as e:
        st.error(f"API error: {e.response.status_code} — {e.response.text}")
        return None

    except Exception as e:
        st.error(f"Unexpected error: {str(e)}")
        return None


def check_backend_health() -> bool:
   
    try:
        r = requests.get(f"{BACKEND_URL}/health", timeout=3)
        return r.status_code == 200 and r.json().get("model_loaded", False)
    except Exception:
        return False


# app layout

# header
st.title("PakWheels Car Price Predictor")
st.markdown(
    "Enter the car's specifications below. The system will predict whether "
    "the car belongs to a **High Price** or **Low Price** category based on "
    "a trained SVM classifier."
)
st.divider()

# backend status checker 

with st.sidebar:
    st.header("⚙️ System Status")
    if check_backend_health():
        st.success("Backend: Connected")
    else:
        st.error("Backend: Not reachable")
        st.code("uvicorn Part2_Task2_backend:app --port 8000 --reload")
    
    st.divider()
    st.markdown("**Model Info**")
    st.markdown("- Algorithm: Support Vector Machine (RBF kernel)")
    st.markdown("- Target: High / Low price (median split ~2.7M PKR)")
    st.markdown("- Features: Year, Engine, Mileage, Transmission, Fuel, Body, City")

# input form
with st.form("prediction_form"):
    st.subheader("Car Specifications")

    col1, col2 = st.columns(2)

    with col1:
        year = st.number_input(
            label    = "Year of Manufacture",
            min_value= 1980,
            max_value= 2024,
            value    = 2018,
            step     = 1,
            help     = "Manufacturing year (1980–2024)",
        )

        engine = st.number_input(
            label    = "Engine Capacity (cc)",
            min_value= 600,
            max_value= 8000,
            value    = 1300,
            step     = 100,
            help     = "Engine displacement in cubic centimetres",
        )

        mileage = st.number_input(
            label    = "Mileage (km)",
            min_value= 0,
            max_value= 500_000,
            value    = 45_000,
            step     = 1000,
            help     = "Odometer reading in kilometres",
        )

        transmission = st.selectbox(
            label   = "Transmission",
            options = TRANSMISSIONS,
            index   = 1,    # default: Automatic
            help    = "Gearbox type",
        )

    with col2:
        fuel = st.selectbox(
            label   = "Fuel Type",
            options = FUELS,
            index   = 0,    # default: Petrol
        )

        body = st.selectbox(
            label   = "Body Type",
            options = BODIES,
            index   = 0,    # default: Sedan
        )

        city = st.selectbox(
            label   = "City",
            options = CITIES,
            index   = 0,    # default: Karachi
        )

    submitted = st.form_submit_button(
        label = "🔍 Predict Price Category",
        use_container_width=True,
        type="primary",
    )

# prediction
if submitted:
    payload = {
        "year"        : int(year),
        "engine"      : float(engine),
        "mileage"     : int(mileage),
        "transmission": transmission,
        "fuel"        : fuel,
        "body"        : body,
        "city"        : city if city != "Other" else "Karachi", 
    }

    with st.spinner("Sending request to model …"):
        result = call_predict_api(payload)

    if result:
        st.divider()
        st.subheader("📊 Prediction Result")

        category = result["price_category"]
        prob_high = result["probability_high"]
        prob_low  = result["probability_low"]
        median    = result["median_price_pkr"]

        if category == "High Price":
            st.success(f"## {category}")
            st.markdown(
                f"This car is likely priced above the median listing price "
                f"of **PKR {median:,.0f}**."
            )
        else:
            st.warning(f"## {category}")
            st.markdown(
                f"This car is likely priced below the median listing price "
                f"of **PKR {median:,.0f}**."
            )

        # confidence scores 
        st.markdown("**Confidence Scores:**")
        col_a, col_b = st.columns(2)
        with col_a:
            st.metric("High Price", f"{prob_high*100:.1f}%")
            st.progress(prob_high)
        with col_b:
            st.metric("Low Price", f"{prob_low*100:.1f}%")
            st.progress(prob_low)

        with st.expander("Raw API Response"):
            st.json(result)

# footer 
st.divider()


Overwriting app.py


In [18]:
import subprocess

# This starts the streamlit server in the background
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0'])
print("Streamlit server started!")

Streamlit server started!




In [ ]:
# 1. Install the tunneling tool
!npm install -g localtunnel

# 2. Start the tunnel
# This will generate a link ending in .loca.lt
# Click that link to open your app!
!lt --port 8501


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.86.1.64:8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸your url is: https://thin-cats-admire.loca.lt


In [13]:
!curl ipv4.icanhazip.com

34.86.1.64
